<a href="https://colab.research.google.com/github/Aditi0912dec/LFBC/blob/main/fedova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KALI

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')



Mounted at /content/gdrive


In [ ]:
drive.mount("/content/gdrive", force_remount=True)

Mounted at /content/gdrive


In [ ]:
import sklearn
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import KFold, train_test_split
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from sklearn.neural_network import MLPClassifier
from keras import models
from tensorflow.keras.utils import to_categorical
import time
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
import csv
import os
import random
from keras.datasets import fashion_mnist
from sklearn.utils import shuffle
from keras.datasets import cifar10



In [ ]:
!pip install pyyaml h5py  # Required to save models in HDF5 format

In [ ]:
# file CNN_model_define
def get_CNN_model():
  model = Sequential()
  model.add(Conv2D(16, (5, 5), activation='relu', kernel_initializer='he_uniform', input_shape=(28, 28, 1)))

  model.add(MaxPooling2D((2,2)))
  model.add(Conv2D(32, (5, 5), activation='relu', kernel_initializer='he_uniform'))

  model.add(MaxPooling2D((2,2)))
  model.add(Flatten())
  #model.add(Dense(1024, activation='relu', kernel_initializer='he_uniform'))
 # model.add(Dense(512, activation='relu', kernel_initializer='he_uniform'))
  model.add(Dense(2, activation='softmax'))
	# compile model
	#opt = SGD(learning_rate=0.01, momentum=0.9)
  model.compile(optimizer='SGD', loss='categorical_crossentropy', metrics=['accuracy'])
  return model

In [ ]:
demo=get_CNN_model()
demo.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 16)        416       
                                                                 
 max_pooling2d (MaxPooling2  (None, 12, 12, 16)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 32)          12832     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 4, 4, 32)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 512)               0         
                                                                 
 dense (Dense)               (None, 2)                 1

# write in json file

# read and write

In [ ]:
# file 2  read_write_create_dir

import os
import csv


def read_from_txt(txt_path,beta_value):
  file= open(txt_path,"r")
  file_split=file.read().split()
  b=[]
  client_key_map=[]
  count=0
  for j in range(0,beta_value):
      for i in range(0,len(file_split)):
        count=count+1
        b.append(int(file_split[i]))
        #print(b)
        if count%beta_value==0:
          client_key_map.append(b)
          b=[]


  print(client_key_map)
  return client_key_map


#print()


def write_to_txt(client_key_list,txt_path,beta_value):
  with open (txt_path, 'w') as file:
      for i in range(0,len(client_key_list)):
        for j in range(0,beta_value):
          file.write(str(client_key_list[i][j]))
          file.write(" ")
        file.write('\n')
  file.close()
  file= open(txt_path,"r")
  file_split=file.read()
  print(file_split)



# to make the directory and sub directory file

def make_dir(n_devices,parent_dir):
  for client in range(0,n_devices):
      print(f"directory for client: {client}")
      client_dir=f"client_{client}"
      client_dir_path=os.path.join(parent_dir,client_dir)
      if os.path.exists(client_dir_path):
        break
      os.mkdir(client_dir_path)
 ## my version decision tree



# initialize key

In [ ]:

# file key_ininialize
def initialize_key(device_list,client_clf_list,beta_value):
  for i in range(0,len(device_list)):
    for j in range(0,beta_value):
      device_list[i].key.append(client_clf_list[i][j])
  return device_list

# Load MNIST digits

In [ ]:
# file  get_dataset_labels
def load_dataset():
  print("data loading")
  (X,y),( X_test,y_test) = mnist.load_data()
 # (X,y),( X_test,y_test)=fashion_mnist.load_data()
  print(X.shape,y.shape,X_test.shape,y_test.shape)
  return X,y,X_test,y_test


def data_processing(X):
  # pixel_value in [0,1]
  print("data processing")
  X = X.reshape((X.shape[0],-1))# converting to 1D array
  X = X/255# normalize the input
  print(X.shape)

  return X


def get_categories(y):
    labels=np.unique(y)
    print("number of classes in the dataset: "+str(len(labels)))
    print("classes in the dataset: "+str(labels))

    return labels

In [ ]:
# file  get_dataset_labels
def load_dataset():
  print("data loading")
  (X,y),( X_test,y_test) = cifar10.load_data()
 # (X,y),( X_test,y_test)=fashion_mnist.load_data()
  print(X.shape,y.shape,X_test.shape,y_test.shape)
  return X,y,X_test,y_test


def data_processing(X):
  # pixel_value in [0,1]
  print("data processing")
  X = X.reshape((X.shape[0],-1))# converting to 1D array
  X = X/255# normalize the input
  print(X.shape)

  return X


def get_categories(y):
    labels=np.unique(y)
    print("number of classes in the dataset: "+str(len(labels)))
    print("classes in the dataset: "+str(labels))

    return labels

# make dir

# data distribution

In [ ]:
# get_distribution
import random
import math
def get_datamap(X,y,labels):
  print("split data as per y label")
  digit_map = {}


  for i in range(len(labels)):
    indices = (y[:,0]==i)
    print(indices)
    Xi = X[indices]
    yi = y[indices]
    digit_map[i] = (Xi,yi)

  for a in range(10):
    Xa,ya = digit_map[a]
    print (Xa.shape, ya.shape)

  return digit_map


def noniid_distribuition(digit_map,device_list,label,beta_value):
  print("In non iid distribution part 1")
  device_index = 0
  partition_per_class=math.floor((len(device_list)*beta_value)/len(label))
  not_avail=set()
  idx=set(np.arange(0,len(device_list)))
  # getting the keys executing for the first time
  for i in range(0,len(digit_map)):
    print(f"dataset: {i}")
    m=0
    Xi,yi = digit_map[i]
    #print(Xi.shape,yi.shape)
    if(i>=0):
      for d in idx:
        if len(device_list[d].key)==beta_value:
          not_avail.add(d)

    idx=idx.difference(not_avail)
    #print(len(idx))
  #  print(f"selected devices : {sel_device_index}")
    # print(f"left : {len(idx)}")
    # print(f"not avail : {not_avail}")
    # print(f"count not avail : {len(not_avail)}")
    #print(len(not_avail))
    #print(len(idx)+len(not_avail))
  # print(len(idx))
  # sel_device_index=random.choices(list(idx),weights=None,k=partition_per_class)
    if(len(idx)<partition_per_class):
      partition_per_class=len(idx)
    if len(idx)==1:
      break
    sel_device_index=random.sample(list(idx),partition_per_class)
    kf = KFold(n_splits=partition_per_class,shuffle=True)
    for train_indices, test_indices in kf.split(Xi):
    #  print("in csv write ")
  # for s in sel_device_index :
      Xa= Xi[test_indices]
      device_list[sel_device_index[m]].add_data_csv(digit=i,X=Xa)
      device_list[sel_device_index[m]].add_key(digit=i)
      print(f"cient id {sel_device_index[m]} and dataset shape {Xa.shape}")
      m=m+1

################################################
  class_label=list(label)
  print("In non iid distribution part 2")
  if len(idx)==1:
    digit_choice=[x for x in label if x not in set(device_list[i].key)]
    m=random.choice(digit_choice)
    index_0=next(iter(idx))
    device_list[index_0].add_data_csv(digit=m,X=Xi)
    device_list[index_0].add_key(digit=m)
  else:

    for i in idx:
      while len(device_list[i].key)<beta_value:
        digit_choice=[x for x in label if x not in set(device_list[i].key)]
        m=random.choice(digit_choice)
        Xi,yi=digit_map[m]
        kf = KFold(n_splits=partition_per_class,shuffle=True)
        for train_indices, test_indices in kf.split(Xi):
          #device_list[i].add_data(digit=m,X=Xi[test_indices])
          device_list[i].add_data_csv(digit=m,X=Xi)
          device_list[i].add_key(digit=m)
          break

######################################################
  #print("device keys")
  for i in range(0, n_devices):
    #print(device_list[i].key)
    print(f"device {i} keys are :{device_list[i].key}")

  return device_list
#######################################################



In [ ]:
# file client_key_mapping
def client_key_map(device_list):
  print("get the client key map")
  client_key_map=[]

  for i in range(0,len(device_list)):
      client_key_map.append(device_list[i].key)
  print(client_key_map)
  return client_key_map

# create h5 file

# get class

In [ ]:
# file class_module
class MyDigitDevice:

  def __init__(self,id=None):
    self.id = id
    print("device id:" +str(id))
    self.key=[]
    self.digit_data = {}
    self.digit_clf = {}
    self.digit_dataset={}
    self.digit_csv={}


  def add_key(self,digit=None):
    self.key.append(digit)

  def add_data_csv(self,digit=None,X=None):
   # print("in add data")
   # print("id" +str(self.id))
    #print("digit:" +str(digit))
    sub_dir=f"client_{self.id}"
    sub_sub_dir=f"digit_{digit}.csv"
    path=os.path.join(parent_dir,sub_dir,sub_sub_dir)
   # print(path)
    with open(path,"w") as csvfile:
      np.savetxt(csvfile,X,delimiter=",")
      self.digit_csv[digit]=X
    csvfile.close()

# how to get y?? or y is the digit

  def add_data(self,digit=None):
     sub_dir=f"client_{self.id}"
     sub_sub_dir=f"digit_{digit}.csv"
     path=os.path.join(parent_dir,sub_dir,sub_sub_dir)
     #print(path)
     X= np.loadtxt(path, delimiter=",")
     y=np.repeat(digit,X.shape[0])
     self.digit_data[digit]=(X,y)

    #self.digit_data[digit] = (X,y)

  def get_training_dataset(self,sel_digit=None,row=None,col=None,s=None,method=None):
    Xcat_list = []
    ycat_list = []
    X_sub=[]
    y_sub=[]
    # print("in get dataset")
    if sel_digit in self.digit_data.keys():
        client_label=list(self.digit_data.keys())
        for i in client_label:
          Xj, yj = self.digit_data[i]# 10 parts of data with each client
       #   print(Xj.shape)
          Xj=Xj.reshape((Xj.shape[0],28,28))
          #print(i)
          Xcat_list.append(Xj)
          ycat_list.append(yj)

    Xcat = np.concatenate(Xcat_list)
    ycat = np.concatenate(ycat_list)
    sel_pos = (ycat==sel_digit)
    Xpos = Xcat[sel_pos].copy()
    ypos = np.ones(ycat[sel_pos].shape)
    if method=="synthetic":
      partial_img=np.full((28,28),0.0)
      # partial_img=np.full((28,28),0.0)
      for k in range(Xpos.shape[0]):
          X=Xpos[k]
          for i in row:
              for j in col:
                partial_img[i:i+s,j:j+s]=X[i:i+s,j:j+s]
                #partial_img=partial_img.reshape(1,784)
                #  print(partial_img.shape)
                  #X.reshape((X.shape[0],-1))
                X_sub.append(partial_img)
              #   count=count+1
                y_sub.append(0)
                partial_img=np.full((28,28),0.0)
          X_sub_list=np.stack(X_sub)
        #  X_sub=Xpos*(np.random.rand(*Xpos.shape)>0.1)
        #  y_sub=np.zeros(ycat[sel_pos].shape)

    X_inv = 1 - Xpos.copy()
    y_inv = 1 - ypos.copy()

    if len(client_label)==1 and method=="synthetic":
      # print("in length 1")
      X = np.concatenate((Xpos,X_inv,X_sub_list))
      y = np.concatenate((ypos,y_inv,y_sub))

    else:

      sel_neg = (ycat!=sel_digit)
      Xneg = Xcat[sel_neg].copy()
      yneg = np.zeros(ycat[sel_neg].shape)
      if method=="synthetic":
        X = np.concatenate((Xpos,Xneg,X_inv,X_sub_list))
        y = np.concatenate((ypos,yneg,y_inv,y_sub))
      else:
        X = np.concatenate((Xpos,Xneg))
        y = np.concatenate((ypos,yneg))
    self.digit_dataset[sel_digit]=(X,y)


  def prepare_clf(self,sel_digit=None,central_model=None):

        central_model_weights=central_model.get_weights()
        X,y=self.digit_dataset[sel_digit]
        X, y = shuffle(X, y, random_state=0)
        y= to_categorical(y,2)
        local_model=self.digit_clf[sel_digit]
        local_model.set_weights(central_model_weights)
        epochs=5
        local_model.compile(optimizer='SGD',loss= 'categorical_crossentropy', metrics=['accuracy'])
        local_model.fit(x=X, y=y, batch_size=15, epochs=epochs, verbose=0)
        sub_dir=f"client_{self.id}"
        sub_sub_dir=f"digit_{sel_digit}.h5"
        h5_path=os.path.join(parent_dir,sub_dir,sub_sub_dir)
        local_model.save(h5_path)
      # self.digit_dataset[sel_digit]=X,y
        self.digit_clf[sel_digit] = local_model




In [ ]:
# file create_device_and_init_model
def make_device(n_devices):
  device_list = []
  model_list = []
  for i in range(n_devices):
    mydevice = MyDigitDevice(id=i)
  # print(getattr(mydevice,'id'))
  # break
    device_list.append(mydevice)
    model_list.append(get_CNN_model())
  return device_list, model_list


# add data

In [ ]:
# to get the data added to the class
# file 3: add data
def add_client_data(device_list,client_key_map,beta_value):
    print("In add data for the clients")
    for i in range(0,len(device_list)):
      print("data added for clients : " +str(i))
      for j in range(0,beta_value):
      #print(client_key_map[i][j])
        device_list[i].add_data(digit=client_key_map[i][j])
      # if i==2:
      #   break
      #device_list[i].add_data(digit=client_key_map[i][1])
    print("The digit data key for the clients")
    for i in range(0,len(device_list)):

      print(device_list[i].digit_data.keys())
    return device_list

## Build one-vs-rest classifier for each digit on each device

In [ ]:
# build one class classifier on each device
# file 4 : prepare classifier
def prepare_client_training_dataset_and_clf(device_list,label,name):
  print("Training dataset for the clients")
  label_device_map = { new_list: [] for new_list in range(len(label)) }

  #print (id_clf_map)
  rowlist=[k for k in range(0,28,14)]
  collist=[k for k in range(0,28,14)]
  stride=14

  for i in range(len(device_list)):
    print("training dataset prepared for client :" +str(i))
    # if i==2:
    #   break
    my_device = device_list[i]
    client_label_list= my_device.digit_data.keys()
    for j in client_label_list:
    # print(j)
     # my_device.get_training_dataset(sel_digit=j,row=rowlist,col=collist,s=stride)
     # X,y=my_device.digit_dataset[j]
      clf=get_CNN_model()
      my_device.digit_clf[j]=clf
      my_device.get_training_dataset(sel_digit=j,row=rowlist,col=collist,s=stride,method=name)
      #my_device.prepare_clf(sel_digit=j,X=X,y=y)# digit wise classifier for each client
     # print(f"device : {my_device.id}")
      label_device_map[j].append(my_device.id)
  return label_device_map


# binarize

In [ ]:
# file 5  binarize
def prepare_binarize_dataset(yb,selected_digit):
    #yb=np.array(yb)
    y_binarize=yb.copy()
    sel_pos=np.where(y_binarize==selected_digit)[0]
    #print(sel_pos)
    sel_neg=np.where(y_binarize!=selected_digit)[0]
    #print(sel_neg)
    for i in sel_pos:
      y_binarize[i]=1
    for i in sel_neg:
      y_binarize[i]=0

    return y_binarize



# initialize central model

In [ ]:
# file central_model_init
def init_central_model(labels):
  central_model_list=[]
  for i in range(len(labels)):
    fedavg_central_model=get_CNN_model()
    fedavg_central_model.compile(loss='categorical_crossentropy', optimizer='SGD', metrics=['accuracy'])
    central_model_list.append(fedavg_central_model)
  return central_model_list

# load local models

In [ ]:
# load_models
def load_local_models(central_model,selected_device,device_list,label,model_list):#device list has index of classiofiers
  trained_local_models=[]
  dataset_cardinality_list=[]
  print(selected_device)
  #print(f"load model for label {label}")
  for i in selected_device:
     # print(type(i))
      #print(device_list[int(26)])
      my_device=device_list[i]
      print(my_device.id)
      new_model=model_list[i]
      f"local_model_for_client_{my_device.id}_digit_{i}.h5"
      sub_dir=f"client_{ my_device.id}"
      sub_sub_dir=f"digit_{label}.h5"
      h5_path=os.path.join(parent_dir,sub_dir,sub_sub_dir)

      dataset_cardinality=my_device.prepare_clf(label,central_model)# digit wise classifier for each client
      dataset_cardinality_list.append(dataset_cardinality)
      new_model= tf.keras.models.load_model(h5_path)
      #trained_local_models.append(my_device.digit_clf[label])
      trained_local_models.append(new_model)

  return trained_local_models

# weight ensemble

In [ ]:
# averaging
def model_weight_ensemble(members,central_model,i):
    n_layers = len(members[0].get_weights())
    # create an set of average model weights
    avg_model_weights = list()
    for layer in range(n_layers):
      #print(layer)
      # collect this layer from each model
      layer_weights = np.array([model.get_weights()[layer] for model in members])
      # weighted average of weights for this layer
      avg_layer_weights = np.average(layer_weights, axis=0)
      # store average layer weights
      avg_model_weights.append(avg_layer_weights)

    #central_model.compile(loss='categorical_crossentropy', optimizer='SGD', metrics=['accuracy'])
    central_model.set_weights(avg_model_weights)# setting the weights of the central model as the average weight of the local model
   # central_model.save(f"central_model_{i}.h5")
    #print(f"ensemble of weights done for label {i}")
    return central_model

In [ ]:
def select_client_per_round(label_device_map):
  selected_device=[]
  for i in range(0,1):
    for i in range(0, len(label_device_map)):
      device=random.sample(label_device_map[i],k=1)
      selected_device.append(device)
  return selected_device

# update the central model

In [ ]:
# update_central

def update_central_model(selected_device,central_model,device_list,model_list):
  global m
  print(selected_device)
  new_central_model_list=[]
  for i in range(0,10):
    print(f"m value:{m}")
    print(selected_device[i])
    members=load_local_models(central_model[i],selected_device[i],device_list,i,model_list)
    m=m+1
    # for j in range(0,len(members)):
    #   label_device_map[i][j].digit_clf=members[j]
    new_central_model_list.append(model_weight_ensemble(members,central_model[i],i))


  return new_central_model_list

# prediction

In [ ]:
# make_prediction
def get_prediction(central_model_list,X_test=None,y_test=None):
  #print("In prediction")
  y_prediction=[]
  avrg_accuracy=0
  for i in range(0,len(central_model_list)):
    y_test_binarize=prepare_binarize_dataset(y_test,i)
    y_prob_pred=central_model_list[i].predict(X_test,verbose=0)# central model pred
    y_pred=[np.argmax(y_prob_pred[i][1]) for i in range(0,len(y_prob_pred))]# to get label specific acc
  #  corr_pred=sum(1 for k,j in zip(y_pred,y_test_binarize) if k==j)# label specific acc
  #  accuracy=round(corr_pred*100/y_test.shape[0],3)
    #print(f"accuracy for central classifier {i} is {accuracy}")
   # avrg_accuracy=avrg_accuracy+accuracy
    y_prediction.append(y_prob_pred)
 # print(f"average accuracy:{avrg_accuracy}")
 # print(len(central_model_list))
  #print(f"mean accuracy of all the clf: {round(avrg_accuracy/len(central_model_list),3)}")
  return y_prediction

# testing

In [ ]:
# test_central
def testing(y_prediction,y_test,round):
  label=[0,1,2,3,4,5,6,7,8,9]
  corr_pred=0
 # print("In testing")
  for i in range(y_test.shape[0]):
    result=[sublist[i] for sublist in y_prediction]
    final_pred=np.argmax(result)
    if final_pred==y_test[i]:
      corr_pred=corr_pred+1
  print(f"accuracy in round {round+1} is :{(corr_pred*100)/y_test.shape[0]}")
  #return corr_pred*100/y_test.shape[0]


# fedova

In [ ]:
from statistics import mode

In [ ]:
# call_fedova
def fedova(select_client,central_model_list,X_test,y_test,result_txt,model_list,device_list,title):
  round=range(0,1)
  accuracy_list=[]
  avrg_acc_list=[]
  print(select_client)
  with open(result_txt,"a") as f:
    f.write(title)
    f.write('\n')
    for i in round:
        print(f"round {i+1}")

        f.write("round: "+str(i+1))
        f.write('\n')
        new_central_model_list= update_central_model(select_client,central_model_list,device_list,model_list)
        y_pred=get_prediction(new_central_model_list,X_test,y_test)
       # avrg_acc_list.append(avrg_accuracy)
        #accuracy=testing(y_pred,y_test,i)
        accuracy_list.append(testing(y_pred,y_test,i))
       # one_cycle_acc=one_cycle_fedova(X_test,label_device_map,select_client)
       # dataframe.loc[len(dataframe.index)]=[accuracy_list[i],avrg_accuracy]
        f.write("accuarcy of multiclass classifier:"+str(accuracy_list[i]))
        f.write('\n')
      #  f.write("aacuracy bt majority voting: "+str(one_cycle_acc))
        f.write('\n')
        dataframe.to_csv(dataframe_path,index=False)
        # if accuracy_list[i]>70.00:
        #  break
        central_model_list=new_central_model_list
 # return central_model_list

# main

In [ ]:
n_devices=10
beta_value=2
parent_dir="/content/gdrive/MyDrive/dataset"
result_txt='/content/gdrive/MyDrive/demo.txt'
key_path='/content/gdrive/MyDrive/client_key.txt'
syn_dataframe='/content/gdrive/MyDrive/syn.csv'
non_syn_dataframe='/content/gdrive/MyDrive/non_syn.csv'
#json_path='/content/gdrive/MyDrive/client_key.json'


test_X_dir='/content/gdrive/MyDrive/test_X.csv'
test_y_dir='/content/gdrive/MyDrive/test_y.csv'

In [ ]:
X,y,X_test,y_test=load_dataset()

X_val,X,y_val,y=train_test_split(X,y,test_size=0.1,random_state=22)

data loading
11490434/11490434 [==============================] - 1s 0us/step
(60000, 28, 28) (60000,) (10000, 28, 28) (10000,)


In [ ]:
X=data_processing(X)

data processing
(6000, 784)


In [ ]:
X_test=data_processing(X_test)


data processing
(10000, 784)


In [ ]:
with open(test_X_dir,"w") as csvfile:
  np.savetxt(csvfile,X_test,delimiter=",")
csvfile.close()

In [ ]:
with open(test_y_dir,"w") as csvfile:
  np.savetxt(csvfile,y_test,delimiter=",")
csvfile.close()

In [ ]:
labels=get_categories(y)

number of classes in the dataset: 10
classes in the dataset: [0 1 2 3 4 5 6 7 8 9]


In [ ]:
make_dir(n_devices,parent_dir)

directory for client: 0


In [ ]:
y

array([7, 0, 9, ..., 9, 8, 1], dtype=uint8)

In [ ]:
digit_map=get_datamap(X,y,labels)


split data as per y label
(625, 784) (625,)
(686, 784) (686,)
(606, 784) (606,)
(594, 784) (594,)
(514, 784) (514,)
(557, 784) (557,)
(564, 784) (564,)
(642, 784) (642,)
(619, 784) (619,)
(593, 784) (593,)


In [ ]:
device_list,model_list=make_device(n_devices)

device id:0
device id:1
device id:2
device id:3
device id:4
device id:5
device id:6
device id:7
device id:8
device id:9


In [ ]:
device_list=noniid_distribuition(digit_map,device_list,labels,beta_value)

In non iid distribution part 1
dataset: 0
In non iid distribution part 2
device 0 keys are :[0, 5]
device 1 keys are :[3, 6]
device 2 keys are :[1, 8]
device 3 keys are :[3, 4]
device 4 keys are :[0, 2]
device 5 keys are :[4, 8]
device 6 keys are :[5, 7]
device 7 keys are :[3, 8]
device 8 keys are :[6, 7]
device 9 keys are :[1, 2]


In [ ]:
client_key_list=list(client_key_map(device_list))


get the client key map
[[0, 5], [3, 6], [1, 8], [3, 4], [0, 2], [4, 8], [5, 7], [3, 8], [6, 7], [1, 2]]


In [ ]:
write_to_txt(client_key_list,key_path,beta_value)

0 5 
3 6 
1 8 
3 4 
0 2 
4 8 
5 7 
3 8 
6 7 
1 2 



In [ ]:
device_list[0].digit_data.keys()

dict_keys([])

# part 2

In [ ]:
device_list,model_list=make_device(n_devices)
X,y,X_test,y_test=load_dataset()

X_test=data_processing(X_test)
X_test= X_test.reshape((X_test.shape[0],28,28))
label=get_categories(y_test)
client_key_list=read_from_txt(key_path,beta_value)
#client_key_json_obj=read_from_txt(json_path)


device id:0
device id:1
device id:2
device id:3
device id:4
device id:5
device id:6
device id:7
device id:8
device id:9
data loading
(60000, 28, 28) (60000,) (10000, 28, 28) (10000,)
data processing
(10000, 784)
number of classes in the dataset: 10
classes in the dataset: [0 1 2 3 4 5 6 7 8 9]
[[0, 5], [3, 6], [1, 8], [3, 4], [0, 2], [4, 8], [5, 7], [3, 8], [6, 7], [1, 2], [0, 5], [3, 6], [1, 8], [3, 4], [0, 2], [4, 8], [5, 7], [3, 8], [6, 7], [1, 2]]


In [ ]:
device_list=initialize_key(device_list,client_key_list,beta_value)

In [ ]:
for i in range(0,n_devices):
  print(device_list[i].key)

[0, 5]
[3, 6]
[1, 8]
[3, 4]
[0, 2]
[4, 8]
[5, 7]
[3, 8]
[6, 7]
[1, 2]


In [ ]:

device_list=add_client_data(device_list,client_key_list,beta_value)

In add data for the clients
data added for clients : 0
data added for clients : 1
data added for clients : 2
data added for clients : 3
data added for clients : 4
data added for clients : 5
data added for clients : 6
data added for clients : 7
data added for clients : 8
data added for clients : 9
The digit data key for the clients
dict_keys([0, 5])
dict_keys([3, 6])
dict_keys([1, 8])
dict_keys([3, 4])
dict_keys([0, 2])
dict_keys([4, 8])
dict_keys([5, 7])
dict_keys([3, 8])
dict_keys([6, 7])
dict_keys([1, 2])


In [ ]:
import copy

In [ ]:
label_device_map[1]

[2, 9]

In [ ]:
#x,y=device_list[0].digit_dataset[4]

In [ ]:
#x.shape

In [ ]:
central_model_list=init_central_model(labels)

#central_model_list_copy=copy.deepcopy(central_model_list)


In [ ]:
central_model_list_copy=copy.deepcopy(central_model_list)

# fedova testing

In [ ]:

m=0
print("without synthetic data")
name="none"
label_device_map=prepare_client_training_dataset_and_clf(device_list,labels,name)




with synthetic data
Training dataset for the clients
training dataset prepared for client :0
training dataset prepared for client :1
training dataset prepared for client :2
training dataset prepared for client :3
training dataset prepared for client :4
training dataset prepared for client :5
training dataset prepared for client :6
training dataset prepared for client :7
training dataset prepared for client :8
training dataset prepared for client :9


In [ ]:
select_client=select_client_per_round(label_device_map)


In [ ]:
for i in select_client[0]:
 print(device_list[i])

In [ ]:
dataframe=pd.DataFrame(columns=['multiclass_acc','mean single clf'])
dataframe_path=non_syn_dataframe
title="without synthetic data"
fedova(select_client,central_model_list,X_test,y_test,result_txt,model_list,device_list,title)
